# RAG with LCEL

**Retrieval-Augmented Generation (RAG)** answers questions by **retrieving** relevant document chunks, **injecting** them into a prompt, and **generating** a grounded reply. LangChain expresses the full pipeline as a single **LCEL chain** — the same **`Runnable`** interface used for prompts, models, and parsers (**02**, **06**).

This notebook assembles end-to-end RAG: the **canonical chain**, **`format_docs`**, **debug variants** that preserve intermediate state, **empty-context** handling, **conversational RAG** with question condensing (**04**), **streaming** answers (**07**), **structured output** (**05**), batch/async invocation, and offline testing patterns.

Prerequisites: **02. Runnable Protocol and LCEL**, **04. Prompt Templates**, **05. Output Parsers and Structured Output**, **06. LCEL Composition Patterns**, **08–10** (documents through retrievers).

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json
import uuid

import langchain_core

print("langchain-core:", langchain_core.__version__)

---

## 1. What RAG with LCEL Looks Like

RAG is not a single LangChain class — it is a **composition pattern**:

```
user question (str)
       │
       ├──────────────────────────────┐
       │                              │
       ▼                              ▼
  Retriever.invoke()          RunnablePassthrough()
       │                              │
       ▼                              │
  list[Document]                       │
       │                              │
       ▼                              │
  format_docs()                        │
       │                              │
       └──────────► RunnableParallel ◄─┘
                         │
                         ▼  {context, question}
                   ChatPromptTemplate
                         │
                         ▼
                     ChatOpenAI
                         │
                         ▼
                  StrOutputParser()
                         │
                         ▼
                    answer (str)
```

| Stage | Runnable | Input → Output |
|-------|----------|----------------|
| **Retrieve** | `retriever` | `str` → `list[Document]` |
| **Format** | `format_docs` | `list[Document]` → `str` |
| **Parallelize** | `RunnableParallel` | `str` → `{context, question}` |
| **Prompt** | `ChatPromptTemplate` | dict → `ChatPromptValue` |
| **Generate** | `ChatOpenAI` | messages → `AIMessage` |
| **Parse** | `StrOutputParser` | `AIMessage` → `str` |

Because every step is a **Runnable**, the chain supports **`invoke`**, **`stream`**, **`batch`**, **`ainvoke`**, and **`config`** callbacks (**07**, **15**) without rewriting the pipeline.

---

## 2. Build the Standalone Index

Same teaching corpus as **09** and **10** — five chunks about a FastAPI Notes API:

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY)
llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

CORPUS = [
    Document(page_content="The Notes API is built with FastAPI. Routes live under /notes with GET, POST, PUT, DELETE.", metadata={"id": "c1", "source": "api-docs"}),
    Document(page_content="Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling revisions.", metadata={"id": "c2", "source": "db-docs"}),
    Document(page_content="JWT bearer tokens authenticate API requests. Send Authorization: Bearer token header.", metadata={"id": "c3", "source": "security-docs"}),
    Document(page_content="Pytest fixtures share database setup. Use conftest.py for session-scoped engines.", metadata={"id": "c4", "source": "test-docs"}),
    Document(page_content="Alembic autogenerate compares SQLAlchemy models to the live schema and drafts revision files.", metadata={"id": "c5", "source": "db-docs"}),
]

vectorstore = Chroma.from_documents(
    documents=CORPUS,
    embedding=embeddings,
    collection_name=f"rag_lcel_{uuid.uuid4().hex[:8]}",
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("indexed:", vectorstore._collection.count(), "documents")

---

## 3. RAG Prompt Design

A production RAG prompt separates **behavior rules**, **context**, and **question**. Rules reduce hallucination when retrieval misses or returns noise (**04. Prompt Templates**):

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

RAG_SYSTEM = """You answer questions using ONLY the context below.
If the answer is not in the context, say "I don't know."
Cite chunk ids in square brackets when possible (e.g. [c3]).
Do not invent API endpoints, commands, or headers not present in the context.
"""

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", RAG_SYSTEM + "\n\nContext:\n{context}"),
    ("human", "{question}"),
])

preview = rag_prompt.invoke({
    "context": "[c3] JWT bearer tokens use Authorization: Bearer header.",
    "question": "What header carries the JWT?",
})
print(preview.messages[0].content[:200], "...")
print("---")
print(preview.messages[1].content)

### 3.1 Prompt Variables

| Variable | Source in chain | Purpose |
|----------|-----------------|--------|
| **`context`** | `retriever \| format_docs` | Retrieved chunks as one string |
| **`question`** | `RunnablePassthrough()` | Original user query |

Keep variable names aligned with **`RunnableParallel`** keys — mismatched names cause **`KeyError`** at prompt render time.

---

## 4. format_docs — Documents to Context String

Retrievers return **`list[Document]`**; prompts need a **single string**. **`format_docs`** is the standard bridge (**10. Retrievers**):

In [ ]:
def format_docs(docs: list[Document]) -> str:
    if not docs:
        return "(no relevant context found)"
    return "\n\n".join(
        f"[{d.metadata.get('id', '?')}] {d.page_content}" for d in docs
    )


sample = retriever.invoke("JWT authentication header")
print(format_docs(sample))

**Design choices:**

- Prefix **`[id]`** so the model (and structured parsers) can cite sources.
- Return an explicit **empty marker** — never pass an empty string silently.
- Join with **`\n\n`** so chunk boundaries stay visible to the model.

Pipe directly in LCEL: **`retriever | format_docs`**.

---

## 5. The Canonical RAG Chain

The standard LCEL shape from **06. LCEL Composition Patterns**:

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

rag_chain = (
    RunnableParallel(
        context=retriever | format_docs,
        question=RunnablePassthrough(),
    )
    | rag_prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("What header carries the JWT token?")
print(answer)

In [ ]:
for question in [
    "What command applies Alembic migrations?",
    "Where do pytest fixtures for DB setup live?",
    "What HTTP methods does the Notes API support?",
]:
    print(f"Q: {question}")
    print(f"A: {rag_chain.invoke(question)}\n")

### 5.1 Why RunnableParallel?

The chain input is a **bare string** (the question). **`RunnableParallel`** fans out:

1. **`context`** branch — same string goes to the retriever.
2. **`question`** branch — **`RunnablePassthrough()`** forwards the string unchanged.

Both branches receive the **same input** and merge into one dict for the prompt. Without passthrough, you would lose the original question after retrieval.

---

## 6. Inspecting the Pipeline Step by Step

Debug retrieval quality by running stages independently before blaming the LLM:

In [ ]:
question = "How do I run database migrations?"

docs = retriever.invoke(question)
context = format_docs(docs)
messages = rag_prompt.invoke({"context": context, "question": question})

print("Retrieved ids:", [d.metadata["id"] for d in docs])
print("--- context ---")
print(context)
print("--- prompt (system excerpt) ---")
print(messages.messages[0].content[:300], "...")

When answers are wrong, check **recall** (right chunk retrieved?) before **generation** (model ignored context?).

---

## 7. Debug RAG — Preserve Intermediate State

Production debugging and citation UIs need **`context`**, **`question`**, and **`answer`** in one response. Use **`RunnablePassthrough.assign`** (**06**):

In [ ]:
rag_debug_chain = (
    RunnableParallel(
        context=retriever | format_docs,
        question=RunnablePassthrough(),
    )
    | RunnablePassthrough.assign(
        answer=(rag_prompt | llm | StrOutputParser())
    )
)

debug_out = rag_debug_chain.invoke("JWT bearer token header?")
print(json.dumps(debug_out, indent=2))

Return **`debug_out`** to the client: show **`answer`** to the user, log **`context`** for support, display chunk ids as footnotes.

### 7.1 Debug with Raw Documents

Keep **`list[Document]`** before formatting when building citation metadata:

In [ ]:
from langchain_core.runnables import RunnableLambda

rag_citation_chain = (
    RunnableParallel(
        docs=retriever,
        question=RunnablePassthrough(),
    )
    | RunnablePassthrough.assign(
        context=RunnableLambda(lambda d: format_docs(d["docs"])),
        source_ids=RunnableLambda(lambda d: [doc.metadata.get("id") for doc in d["docs"]]),
    )
    | RunnablePassthrough.assign(
        answer=(rag_prompt | llm | StrOutputParser())
    )
)

citation_out = rag_citation_chain.invoke("Alembic upgrade command?")
print("sources:", citation_out["source_ids"])
print("answer:", citation_out["answer"])

---

## 8. Empty Context and Hallucination Guardrails

When retrieval returns **zero** documents (threshold retriever, bad query, wrong filter), the LLM may **hallucinate** unless the prompt and formatter cooperate:

In [ ]:
EMPTY_RAG_SYSTEM = """You answer questions using ONLY the context below.
If the context says "(no relevant context found)" or is empty, you MUST respond:
"I don't know — no relevant documentation was retrieved."
Never guess commands, headers, or routes.
"""

empty_safe_prompt = ChatPromptTemplate.from_messages([
    ("system", EMPTY_RAG_SYSTEM + "\n\nContext:\n{context}"),
    ("human", "{question}"),
])

strict_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 3, "score_threshold": 0.99},
)

empty_context_chain = (
    RunnableParallel(
        context=strict_retriever | format_docs,
        question=RunnablePassthrough(),
    )
    | empty_safe_prompt
    | llm
    | StrOutputParser()
)

print(empty_context_chain.invoke("What is the Kubernetes ingress controller?"))

**Defense in depth:**

1. **`format_docs`** — explicit empty marker string.
2. **Prompt** — instruct refusal on empty context.
3. **Retriever** — score threshold rejects weak matches (**10. Retrievers**).
4. **Optional** — short-circuit before LLM call when `docs` is empty (save cost).

In [ ]:
def rag_or_refuse(question: str) -> str:
    docs = retriever.invoke(question)
    if not docs:
        return "I don't know — no relevant documentation was retrieved."
    return rag_chain.invoke(question)


# Normal path still works:
print(rag_or_refuse("JWT header name?"))

---

## 9. Tuning Retrieval Inside RAG

RAG quality is often a **retrieval** problem. Swap retriever config without changing the rest of the chain:

In [ ]:
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 5, "lambda_mult": 0.5},
)

db_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2, "filter": {"source": "db-docs"}},
)

def build_rag_chain(active_retriever):
    return (
        RunnableParallel(
            context=active_retriever | format_docs,
            question=RunnablePassthrough(),
        )
        | rag_prompt
        | llm
        | StrOutputParser()
    )


mmr_chain = build_rag_chain(mmr_retriever)
db_chain = build_rag_chain(db_retriever)

q = "Tell me about Alembic migrations and autogenerate"
print("MMR:     ", mmr_chain.invoke(q)[:120], "...")
print("DB only: ", db_chain.invoke(q)[:120], "...")

Combine with **routers** (**06**) — classify the question domain, pick the filtered retriever, then run the same RAG template.

---

## 10. Conversational RAG — Condense the Question

Multi-turn chat sends **follow-ups** like "What command do I run?" — useless for retrieval without prior context. **Condense** the follow-up into a **standalone question** before retrieval (**04. Prompt Templates**):

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import MessagesPlaceholder

condense_prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder("history"),
    (
        "human",
        "Given the conversation above, rewrite this follow-up as a standalone question "
        "that can be understood without the chat history.\n"
        "Follow-up: {question}\n"
        "Standalone question:",
    ),
])

condense_chain = condense_prompt | llm | StrOutputParser()

history = [
    HumanMessage(content="Tell me about Alembic migrations."),
    AIMessage(content="Alembic applies SQLAlchemy schema migrations to your database."),
]

standalone = condense_chain.invoke({"history": history, "question": "What command do I run?"})
print("standalone:", standalone)

The standalone question ("What Alembic command applies migrations?") retrieves **`c2`** reliably; the raw follow-up might not.

---

## 11. Full Conversational RAG Chain

Wire condense → retrieve → answer. Input is a **dict** with **`history`** and **`question`**:

In [ ]:
pick_standalone = RunnableLambda(lambda d: d["standalone_question"])

conversational_rag = (
    RunnablePassthrough.assign(
        standalone_question=condense_chain,
    )
    | RunnablePassthrough.assign(
        context=pick_standalone | retriever | format_docs,
    )
    | RunnablePassthrough.assign(
        answer=(rag_prompt | llm | StrOutputParser()),
    )
)

conv_out = conversational_rag.invoke({
    "history": history,
    "question": "What command do I run?",
})

print("standalone:", conv_out["standalone_question"])
print("answer:", conv_out["answer"])

### 11.1 RunnableLambda for Dict Routing

When a step needs a **specific key** from a dict, wrap with **`RunnableLambda`**:

In [ ]:
import operator

conversational_rag_v2 = (
    RunnablePassthrough.assign(standalone_question=condense_chain)
    | RunnableParallel(
        context=RunnableLambda(operator.itemgetter("standalone_question")) | retriever | format_docs,
        question=RunnableLambda(operator.itemgetter("question")),
        standalone_question=RunnableLambda(operator.itemgetter("standalone_question")),
        history=RunnableLambda(operator.itemgetter("history")),
    )
    | RunnablePassthrough.assign(answer=(rag_prompt | llm | StrOutputParser()))
)

out = conversational_rag_v2.invoke({"history": history, "question": "What command do I run?"})
print(out["answer"])

**Note:** Full chat memory patterns (persisting history, trimming windows) are in **14. Memory and Chat History**. Here the focus is the **condense → retrieve → generate** LCEL shape.

---

## 12. Streaming RAG Answers

**`.stream()`** on the chain yields **answer tokens** after retrieval completes (**07. Streaming, Batch, and Async**):

In [ ]:
print("Streaming RAG answer:")
for chunk in rag_chain.stream("What header carries JWT?"):
    print(chunk, end="", flush=True)
print("\n--- done ---")

**UX implication:** retrieval is **blocking** — the user sees nothing until the first token. Show a "Searching documentation…" state during retrieval, then stream the answer.

For **`astream_events`**, wrap the same chain — events fire for retriever, prompt, and model steps (**15. Callbacks, Caching, and Observability**).

---

## 13. Structured RAG Output

User-facing bots often need **`StrOutputParser`**. APIs and eval pipelines may require **typed** responses — **`with_structured_output`** (**05. Output Parsers and Structured Output**):

In [ ]:
from pydantic import BaseModel, Field


class RagAnswer(BaseModel):
    answer: str = Field(description="Answer grounded strictly in context")
    confidence: float = Field(description="0.0-1.0 confidence that context supports the answer")
    source_ids: list[str] = Field(description="Chunk ids used, e.g. c2, c3")


structured_rag = (
    RunnableParallel(
        context=retriever | format_docs,
        question=RunnablePassthrough(),
    )
    | rag_prompt
    | llm.with_structured_output(RagAnswer)
)

structured_result = structured_rag.invoke("What header carries the JWT token?")
print(structured_result)

Structured output is **not token-streamable** by default — stream prose for UX, validate structure after completion, or use provider-specific partial JSON if available.

---

## 14. batch and async RAG

Run the **same chain** over many questions for eval or bulk Q&A (**07**):

In [ ]:
eval_questions = [
    "JWT authentication header?",
    "Alembic upgrade command?",
    "pytest conftest fixtures?",
]

batch_answers = rag_chain.batch(eval_questions)
for q, a in zip(eval_questions, batch_answers):
    print(f"Q: {q}")
    print(f"A: {a[:100]}...\n")

In [ ]:
import asyncio


async def async_rag_demo():
    answer = await rag_chain.ainvoke("FastAPI notes routes?")
    print("ainvoke:", answer)


asyncio.run(async_rag_demo())

Use **`config={"max_concurrency": N}`** on **`batch`** to respect API rate limits.

---

## 15. Testing RAG Without Live API Calls

Unit-test **chain wiring** with a **fake retriever** and **`FakeListChatModel`** (**03**, **10**):

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.runnables import RunnableLambda

FAKE_DOCS = [CORPUS[2]]  # c3 JWT chunk

fake_retriever = RunnableLambda(lambda q: FAKE_DOCS)
fake_llm = FakeListChatModel(
    responses=["Use the Authorization: Bearer header [c3]."]
)

test_rag_chain = (
    RunnableParallel(
        context=fake_retriever | format_docs,
        question=RunnablePassthrough(),
    )
    | rag_prompt
    | fake_llm
    | StrOutputParser()
)

assert "Authorization" in test_rag_chain.invoke("JWT header?")
print("offline RAG test passed")

Test **retrieval** and **generation** separately: golden queries → expected chunk ids; fixed context → expected answer shape.

---

## 16. RAG vs Agents — When to Stop at RAG

| Need | RAG chain | Agent (**12**, **13**) |
|------|-----------|--------------------------|
| Answer from **indexed docs** | Yes | Overkill |
| **Fixed** retrieval → answer flow | Yes | Overkill |
| Call **live APIs**, run code, browse web | No | Yes |
| Multi-step **reasoning with tools** | No | Yes |

RAG excels at **grounded Q&A** over a known corpus. When the model must **decide** which tool to call or iterate, graduate to agents — but keep the RAG subchain as one tool inside the agent.

---

## 17. Production Checklist

| Area | Recommendation |
|------|----------------|
| **Ingest** | Chunk + embed + metadata (**08**, **09**) |
| **Retrieve** | Tune `k`, MMR, filters (**10**) |
| **Prompt** | Grounding rules + empty-context refusal |
| **Debug** | `assign` chain returns context + sources |
| **Latency** | Show retrieval spinner; stream generation |
| **Cost** | Short-circuit empty retrieval; cache embeddings |
| **Observability** | Callbacks / LangSmith (**15**) |
| **Resilience** | Fallback models (**16. Fallbacks and Configurable Runnables**) |
| **Multi-turn** | Condense question or memory (**14**) |

---

## 18. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Missing `question=RunnablePassthrough()` | Prompt `KeyError` | Always parallelize both keys |
| Empty `format_docs` output | Silent hallucination | Explicit empty marker + prompt rule |
| Follow-up without condense | Bad retrieval | Condense chain before retriever |
| `k` too large | Noise in context | Lower k or use MMR/threshold |
| Wrong embedding model | Poor recall | Same model at index and query time |
| No debug assign | Cannot trace bad answers | Keep `context` in output dict |
| Structured output + stream UX | Parsing fails mid-stream | Stream text or wait for full JSON |
| Blaming LLM first | Misdiagnosed failure | Inspect retrieved ids before generation |

---

## 19. Summary

**RAG with LCEL** is the standard **`RunnableParallel(context=retriever | format_docs, question=RunnablePassthrough()) | prompt | llm | parser`** pipeline. Every stage is a Runnable — swap retrievers, prompts, or models without rewriting orchestration code.

Key takeaways:

- **`format_docs`** turns **`list[Document]`** into cited context strings with empty handling.
- **`RunnablePassthrough.assign`** builds **debug** and **citation** responses.
- **Empty retrieval** needs formatter + prompt + optional short-circuit defenses.
- **Conversational RAG** condenses follow-ups before retrieval.
- **Streaming** applies to generation; retrieval remains a pre-step.
- **`with_structured_output`** adds typed answers for APIs and eval.
- **Fake retriever + fake LLM** enable offline chain tests.

Demonstrations built a vector index, wired the canonical chain, debug and citation variants, empty-context guards, conversational condense flow, streaming, structured output, batch/async invocation, and offline testing.

Next: **12. Tools and Tool Calling** — extend beyond static retrieval to model-invoked tools.